In [ ]:
from pathlib import Path
import os
import numpy as np

from marmopose.config import Config
from marmopose.utils.data_io import load_points_3d_h5
session = '260513'
data_dir = Path('/scratch/VideoTracking/Videos') / session
srv_dir = Path('/srv/MarmOT/VideoTracking/Videos') / session
data_dir_etho = data_dir / 'Etho'
data_dir_home = data_dir / 'Home'
output_combined_dir = data_dir / 'Output_combined'
os.makedirs(output_combined_dir, exist_ok=True)

os.chdir('..')
# print(os.listdir('../..'))
etho_dimensions = [660, 560, 800, 30]
home_dimensions = [1200, 815, 900, 30]

config_path = '../configs/default.yaml'

config = Config(
        config_path=config_path,
        n_tracks=1,
        project='../demos/single',
        room_dimensions = etho_dimensions,
        # do_optimize=False,
    )



In [ ]:
points_3d_path = data_dir_etho / 'Output' / 'points_3d_both' / 'optimized.h5'
points_3d_path_home = data_dir_home / 'Output' / 'points_3d_both' / 'optimized.h5'
all_points_3d = load_points_3d_h5(points_3d_path)
all_points_3d_home = load_points_3d_h5(points_3d_path_home)

duration = np.min((all_points_3d.shape[1],all_points_3d_home.shape[1]))
all_points_3d = all_points_3d[0,:duration,:,:]
all_points_3d_home = all_points_3d_home[0,:duration,:,:]
idx_head = config.animal['bodyparts'].index('head')
idx_leftear = config.animal['bodyparts'].index('leftear')
idx_rightear = config.animal['bodyparts'].index('rightear')

idx_neck = config.animal['bodyparts'].index('neck')
idx_spinemid = config.animal['bodyparts'].index('spinemid')
idx_tailbase = config.animal['bodyparts'].index('tailbase')

position_head = np.mean(all_points_3d[:,np.r_[idx_head,idx_leftear,idx_rightear]], axis = 1)
position_body = np.mean(all_points_3d[:,np.r_[idx_neck,idx_spinemid,idx_tailbase]], axis = 1)
position_head_home = np.mean(all_points_3d_home[:,np.r_[idx_head,idx_leftear,idx_rightear]], axis = 1)
position_body_home = np.mean(all_points_3d_home[:,np.r_[idx_neck,idx_spinemid,idx_tailbase]], axis = 1)
print(position_head)


In [ ]:
from marmopose.utils.data_io import get_offset_from_point
import marmopose.utils.plotting as mplt
from importlib import reload
reload(mplt)
from marmopose.utils.plotting import plot_heatmaps, plot_distribution_distance

offset = get_offset_from_point(data_dir_etho / 'Calib')
offset_home = get_offset_from_point(data_dir_home / 'Calib')

fig_head, axs_head = plot_heatmaps(all_points_3d - offset, etho_dimensions, (idx_head, idx_leftear, idx_rightear))
fig_body, axs_body = plot_heatmaps(all_points_3d - offset, etho_dimensions, (idx_neck, idx_spinemid, idx_tailbase))
fig_head, axs_head = plot_heatmaps(all_points_3d_home - offset_home, home_dimensions, (idx_head, idx_leftear, idx_rightear))
fig_body, axs_body = plot_heatmaps(all_points_3d_home - offset_home, home_dimensions, (idx_neck, idx_spinemid, idx_tailbase))

In [ ]:
fig_head, axs_head = plot_distribution_distance(all_points_3d, all_points_3d_home, (idx_head, idx_leftear, idx_rightear))
fig_head, axs_head = plot_distribution_distance(all_points_3d, all_points_3d_home, (idx_spinemid, idx_neck, idx_tailbase))
closest_to_etho = np.copy(all_points_3d_home)
closest_to_etho[..., 0] = offset_home[0]
closest_to_etho[closest_to_etho[..., 1] < offset[1], 1] = offset[1]
closest_to_etho[closest_to_etho[..., 1] > etho_dimensions[1] + offset[1], 1] = etho_dimensions[1] + offset[1]
closest_to_etho[closest_to_etho[..., 2] < 0, 2] = 0
closest_to_etho[closest_to_etho[..., 2] > etho_dimensions[2] + offset[2], 2] = etho_dimensions[2] + offset[2]
closest_to_home = np.copy(all_points_3d)
closest_to_home[..., 0] = offset[0] + etho_dimensions[0]
closest_to_home[closest_to_home[..., 1] < offset_home[1], 1] = offset_home[1]
closest_to_home[closest_to_home[..., 1] > home_dimensions[1] + offset_home[1], 1] = home_dimensions[1] + offset_home[1]
closest_to_home[closest_to_home[..., 2] < offset_home[2], 2] = offset_home[2]
closest_to_home[closest_to_home[..., 2] > home_dimensions[2] + offset_home[2], 2] = home_dimensions[2] + offset_home[2]
fig_head, axs_head = plot_distribution_distance(all_points_3d, closest_to_home, (idx_head, idx_leftear, idx_rightear))
fig_head, axs_head = plot_distribution_distance(all_points_3d, closest_to_home, (idx_spinemid, idx_neck, idx_tailbase))
fig_head, axs_head = plot_distribution_distance(all_points_3d_home, closest_to_etho, (idx_head, idx_leftear, idx_rightear))
fig_head, axs_head = plot_distribution_distance(all_points_3d_home, closest_to_etho, (idx_spinemid, idx_neck, idx_tailbase))


In [ ]:
from marmopose.utils.geometry import points_in_cone

home_sees_etho_path = output_combined_dir / 'home_sees_etho.npy'
etho_sees_home_path = output_combined_dir / 'etho_sees_home.npy'
joint_gaze_path = output_combined_dir / 'joint_gaze.npy'

apex_etho = all_points_3d[:,idx_head,:]
axis_etho = all_points_3d[:,idx_head,:] - (all_points_3d[:,idx_rightear,:] + all_points_3d[:,idx_leftear,:])/2
axis_etho = axis_etho/np.linalg.norm(axis_etho, axis = -1)[:,None]

apex_home= all_points_3d_home[:,idx_head,:]
axis_home = all_points_3d_home[:,idx_head,:] - (all_points_3d_home[:,idx_rightear,:] + all_points_3d_home[:,idx_leftear,:])/2
axis_home = axis_home/np.linalg.norm(axis_home, axis = -1)[:,None]

if os.path.isfile(home_sees_etho_path):
    home_sees_etho = np.load(home_sees_etho_path)
else:
    home_sees_etho = points_in_cone(all_points_3d_home.transpose((1,0,2)), apex_etho, axis_etho, 5 * np.pi/180, 3000)
if os.path.isfile(etho_sees_home_path):
    etho_sees_home = np.load(etho_sees_home_path)
else:
    etho_sees_home = points_in_cone(all_points_3d.transpose((1,0,2)), apex_home, axis_home, 5 * np.pi/180, 3000)
if os.path.isfile(joint_gaze_path):
    joint_gaze = np.load(joint_gaze_path)
else:
    xs = np.arange(-1500,1501,100)
    ys = np.arange(-500,1501,100)
    zs = np.arange(-500,1501,100)
    x_allpoints, y_allpoints, z_allpoints = np.meshgrid(xs,ys,zs)
    allpoints = np.concatenate((x_allpoints.flatten()[:,None,None], y_allpoints.flatten()[:,None,None], z_allpoints.flatten()[:,None,None]), axis = -1)
    home_sees_allpoints = points_in_cone(allpoints, apex_etho, axis_etho, 5 * np.pi/180, 3000)
    etho_sees_allpoints = points_in_cone(allpoints, apex_home, axis_home, 5 * np.pi/180, 3000)
    joint_gaze = np.logical_or.reduce(np.logical_and(home_sees_allpoints,etho_sees_allpoints))

etho_sees_home_ag = np.logical_or.reduce(etho_sees_home, axis = 0)
home_sees_etho_ag = np.logical_or.reduce(home_sees_etho, axis = 0)
etho_sees_home_face_ag = np.logical_or.reduce(etho_sees_home[np.r_[(idx_head, idx_rightear, idx_leftear)],:], axis = 0)
home_sees_etho_face_ag = np.logical_or.reduce(home_sees_etho[np.r_[(idx_head, idx_rightear, idx_leftear)],:], axis = 0)
etho_sees_home_body_ag = np.logical_or.reduce(etho_sees_home[np.r_[(idx_spinemid, idx_neck, idx_tailbase)],:], axis = 0)
home_sees_etho_body_ag = np.logical_or.reduce(home_sees_etho[np.r_[(idx_spinemid, idx_neck, idx_tailbase)],:], axis = 0)




In [ ]:
print(np.sum(np.logical_and.reduce([np.einsum('ij,ij->i',axis_home[2:],axis_home[:-2])>0.997, np.einsum('ij,ij->i',axis_home[:-2],axis_home[1:-1])>0.997, np.einsum('ij,ij->i',axis_home[2:],axis_home[1:-1])>0.997])))
print(np.sum(np.einsum('ij,ij->i',axis_home[4:],axis_home[:-4])>0.997))
print(axis_home.shape)
fixed_gaze_indices = np.nonzero(np.logical_and.reduce([np.einsum('ij,ij->i',axis_home[2:],axis_home[:-2])>0.997, np.einsum('ij,ij->i',axis_home[:-2],axis_home[1:-1])>0.997, np.einsum('ij,ij->i',axis_home[2:],axis_home[1:-1])>0.997]))[0]

In [ ]:
print(fixed_gaze_indices)

In [ ]:
np.arccos(0.99) * 180/np.pi
print(np.cos(5*np.pi/180))

In [ ]:
import matplotlib.pyplot as plt
plt.bar(np.arange(6), np.sum([etho_sees_home_ag, home_sees_etho_ag, etho_sees_home_face_ag, home_sees_etho_face_ag, etho_sees_home_body_ag, home_sees_etho_body_ag],axis = 1)/etho_sees_home_ag.size)
plt.show()
distance_bins = np.arange(0,2501,250)
indices = (idx_head, idx_rightear, idx_leftear)
positions1 = np.mean(all_points_3d[:,np.r_[indices]], axis = 1)
positions2 = np.mean(all_points_3d_home[:,np.r_[indices]], axis = 1)
positions_closest_to_home = np.mean(closest_to_home[:,np.r_[indices]], axis = 1)
positions_closest_to_etho = np.mean(closest_to_etho[:,np.r_[indices]], axis = 1)
distances_head = np.linalg.norm(positions1-positions2, axis = 1)
indices_bins = [np.where(np.logical_and(distances_head >= bin1,distances_head < bin2))[0] for bin1,bin2 in zip(distance_bins[:-1], distance_bins[1:])]
distances_head_closest_to_home = np.linalg.norm(positions2-positions_closest_to_home, axis = 1)
indices_bins_closest_to_home = [np.where(np.logical_and(distances_head_closest_to_home >= bin1,distances_head_closest_to_home < bin2))[0] for bin1,bin2 in zip(distance_bins[:-1], distance_bins[1:])]
distances_head_closest_to_etho = np.linalg.norm(positions1-positions_closest_to_etho, axis = 1)
indices_bins_to_etho = [np.where(np.logical_and(distances_head_closest_to_etho >= bin1,distances_head_closest_to_etho < bin2))[0] for bin1,bin2 in zip(distance_bins[:-1], distance_bins[1:])]

etho_sees_home_ag_per_distance_bin = [np.sum(etho_sees_home_ag[indices])/indices.size for indices in indices_bins]
home_sees_etho_ag_per_distance_bin = [np.sum(home_sees_etho_ag[indices])/indices.size for indices in indices_bins]
etho_sees_home_face_ag_per_distance_bin = [np.sum(etho_sees_home_face_ag[indices])/indices.size for indices in indices_bins]
home_sees_etho_face_ag_per_distance_bin = [np.sum(home_sees_etho_face_ag[indices])/indices.size for indices in indices_bins]
etho_sees_home_body_ag_distance_bin = [np.sum(etho_sees_home_body_ag[indices])/indices.size for indices in indices_bins]
home_sees_etho_body_ag_per_distance_bin = [np.sum(home_sees_etho_body_ag[indices])/indices.size for indices in indices_bins]
plt.plot(distance_bins[1:],etho_sees_home_ag_per_distance_bin)
plt.show()
plt.plot(distance_bins[1:],home_sees_etho_ag_per_distance_bin)
plt.show()
plt.plot(distance_bins[1:],etho_sees_home_face_ag_per_distance_bin)
plt.show()
plt.plot(distance_bins[1:],home_sees_etho_face_ag_per_distance_bin)
plt.show()
plt.plot(distance_bins[1:],etho_sees_home_body_ag_distance_bin)
plt.show()
plt.plot(distance_bins[1:],home_sees_etho_body_ag_per_distance_bin)
plt.show()

In [ ]:
import csv
import pandas as pd
from scipy import stats
led_time = None
distance_bins = np.arange(0,2601,200)

with open(srv_dir / 'arduino_logging.csv') as fp:
    reader = csv.reader(fp)
    for r in reader:
        if r[1] == 'BEEP1.1 ON':
            beep_time = float(r[0])/1000
        elif r[1] == 'LED ON' and led_time is None:
            led_time = float(r[0])/1000
diff_led_beep = led_time - beep_time
calls_ann_dir = Path(f'/srv/MarmOT/ISCMJ/Kurt_baseline_preinj/{session}_homecage/')
voc_Kurt = pd.read_csv(calls_ann_dir / f'{session}_Kurt_sync_annotations.csv')
voc_Kurt['start_seconds'] = voc_Kurt['start_seconds'] - diff_led_beep
voc_Kurt['stop_seconds'] = voc_Kurt['stop_seconds'] - diff_led_beep
voc_Kurt = voc_Kurt.loc[voc_Kurt['start_seconds'] > 0]
voc_Bichon = pd.read_csv(calls_ann_dir / f'{session}_Bichon_sync_annotations.csv')
voc_Bichon['start_seconds'] = voc_Bichon['start_seconds'] - diff_led_beep
voc_Bichon['stop_seconds'] = voc_Bichon['stop_seconds'] - diff_led_beep
voc_Bichon = voc_Bichon.loc[voc_Bichon['start_seconds'] > 0]
time_bins = np.arange(0,positions1.shape[0]*0.04,0.04)
time_increments = np.arange(-1500,1500)

trills_Kurt = voc_Kurt.loc[voc_Kurt['name'] == 'trill']
trills_Bichon = voc_Bichon.loc[voc_Bichon['name'] == 'trill']
phees_Kurt = voc_Kurt.loc[voc_Kurt['name'] == 'phee']
phees_Bichon = voc_Bichon.loc[voc_Bichon['name'] == 'phee']
twitters_Kurt = voc_Kurt.loc[voc_Kurt['name'] == 'twitter']
twitters_Bichon = voc_Bichon.loc[voc_Bichon['name'] == 'twitter']
print("Trills Kurt:", len(trills_Kurt))
print("Trills Bichon:", len(trills_Bichon))
print("Phees Kurt:", len(phees_Kurt))
print("Phees Bichon:", len(phees_Bichon))
print("Twitters Kurt:", len(twitters_Kurt))
print("Twitters Bichon:", len(twitters_Bichon))

trills_home_start_binned = np.digitize(trills_Kurt['start_seconds'],time_bins)
trills_home_start = np.zeros((positions1.shape[0]))
trills_home_start[trills_home_start_binned] = 1
trills_home_start_1m = np.nonzero(trills_home_start_binned)[0][:,None] + time_increments

trills_etho_start_binned = np.digitize(trills_Bichon['start_seconds'],time_bins)
trills_etho_start = np.zeros((positions1.shape[0]))
trills_etho_start[trills_etho_start_binned] = 1
trills_home_start_1m = trills_etho_start_binned[:,None] + time_increments


twitters_home_start_binned = np.digitize(twitters_Kurt['start_seconds'],time_bins)
twitters_home_end_binned = np.digitize(twitters_Kurt['stop_seconds'],time_bins)
twitters_home_start = twitters_home_end = np.zeros((positions1.shape[0]))
twitters_home_start[twitters_home_start_binned] = 1
twitters_home_end[twitters_home_end_binned] = 1

twitters_etho_start_binned = np.digitize(twitters_Bichon['start_seconds'],time_bins)
twitters_etho_end_binned = np.digitize(twitters_Bichon['stop_seconds'],time_bins)
twitters_etho_start = twitters_etho_end = np.zeros((positions1.shape[0]))
twitters_etho_start[twitters_etho_start_binned] = 1
twitters_etho_end[twitters_etho_end_binned] = 1

phees_home_start_binned = np.digitize(phees_Kurt['start_seconds'],time_bins)
phees_home_end_binned = np.digitize(phees_Kurt['stop_seconds'],time_bins)
phees_home_start = phees_home_end = np.zeros((positions1.shape[0]))
phees_home_start[phees_home_start_binned] = 1
phees_home_end[phees_home_end_binned] = 1

phees_etho_start_binned = np.digitize(phees_Bichon['start_seconds'],time_bins)
phees_etho_end_binned = np.digitize(phees_Bichon['stop_seconds'],time_bins)
phees_etho_start = phees_etho_end = np.zeros((positions1.shape[0]))
phees_etho_start[phees_etho_start_binned] = 1
phees_etho_end[phees_etho_end_binned] = 1

distance_bins = np.arange(0,2501,250)

trill_etho_start_per_distance_bin = [np.sum(trills_etho_start[indices])/indices.size for indices in indices_bins]
trill_home_start_per_distance_bin = [np.sum(trills_home_start[indices])/indices.size for indices in indices_bins]
twitter_etho_start_per_distance_bin = [np.sum(twitters_etho_start[indices])/indices.size for indices in indices_bins]
twitter_home_start_per_distance_bin = [np.sum(twitters_home_start[indices])/indices.size for indices in indices_bins]
phee_etho_start_per_distance_bin = [np.sum(phees_etho_start[indices])/indices.size for indices in indices_bins]
phee_home_start_per_distance_bin = [np.sum(phees_home_start[indices])/indices.size for indices in indices_bins]
plt.plot(distance_bins[1:],twitter_etho_start_per_distance_bin)
plt.plot(distance_bins[1:],trill_etho_start_per_distance_bin)
plt.plot(distance_bins[1:],phee_etho_start_per_distance_bin)
plt.show()
plt.plot(distance_bins[1:],twitter_home_start_per_distance_bin)
plt.plot(distance_bins[1:],trill_home_start_per_distance_bin)
plt.plot(distance_bins[1:],phee_home_start_per_distance_bin)
plt.show()



def mean_ci_distance_post_trill(trills_start_binned, distances):
    trills_1m = trills_start_binned[:,None] + time_increments
    distance_around_trill = distances[trills_1m]
    distance_around_trill = distance_around_trill[~np.any(distance_around_trill > 3000, axis=1), :]
    mean= np.mean(distance_around_trill, axis=0)
    ci = stats.sem(distance_around_trill, axis=0, ddof=1) * 1.96
    return mean, ci

mean_distance_post_trill_home, ci_distance_post_trill_home = mean_ci_distance_post_trill(trills_home_start_binned,distances_head)
mean_distance_post_trill_etho, ci_distance_post_trill_etho = mean_ci_distance_post_trill(trills_etho_start_binned,distances_head)
mean_distance_closest_home_post_trill_home, ci_distance_closest_home_post_trill_home = mean_ci_distance_post_trill(trills_home_start_binned,distances_head_closest_to_home)
mean_distance_closest_home_post_trill_etho, ci_distance_closest_home_post_trill_etho = mean_ci_distance_post_trill(trills_etho_start_binned,distances_head_closest_to_home)
mean_distance_closest_etho_post_trill_home, ci_distance_closest_etho_post_trill_home = mean_ci_distance_post_trill(trills_home_start_binned,distances_head_closest_to_etho)
mean_distance_closest_etho_post_trill_etho, ci_distance_closest_etho_post_trill_etho = mean_ci_distance_post_trill(trills_etho_start_binned,distances_head_closest_to_etho)

plt.plot(time_increments,mean_distance_post_trill_home)
plt.fill_between(time_increments, mean_distance_post_trill_home - ci_distance_post_trill_home, mean_distance_post_trill_home + ci_distance_post_trill_home, alpha=0.5)
plt.plot(time_increments,mean_distance_post_trill_etho)
plt.fill_between(time_increments, mean_distance_post_trill_etho - ci_distance_post_trill_etho, mean_distance_post_trill_etho + ci_distance_post_trill_etho, alpha=0.5)
plt.show()

plt.plot(time_increments,mean_distance_closest_home_post_trill_home)
plt.fill_between(time_increments, mean_distance_closest_home_post_trill_home - ci_distance_closest_home_post_trill_home, mean_distance_closest_home_post_trill_home + ci_distance_closest_home_post_trill_home, alpha=0.5)
plt.plot(time_increments,mean_distance_closest_home_post_trill_etho)
plt.fill_between(time_increments, mean_distance_closest_home_post_trill_etho - ci_distance_closest_home_post_trill_etho, mean_distance_closest_home_post_trill_etho + ci_distance_closest_home_post_trill_etho, alpha=0.5)
plt.show()


plt.plot(time_increments,mean_distance_closest_etho_post_trill_home)
plt.fill_between(time_increments, mean_distance_closest_etho_post_trill_home - ci_distance_closest_etho_post_trill_home, mean_distance_closest_etho_post_trill_home + ci_distance_closest_etho_post_trill_home, alpha=0.5)
plt.plot(time_increments,mean_distance_closest_etho_post_trill_etho)
plt.fill_between(time_increments, mean_distance_closest_etho_post_trill_etho - ci_distance_closest_etho_post_trill_etho, mean_distance_closest_etho_post_trill_etho + ci_distance_closest_etho_post_trill_etho, alpha=0.5)
plt.show()

In [ ]:
w = 5
trills_home_start_1m = trills_home_start_binned[:,None] + time_increments
trills_etho_start_1m = trills_etho_start_binned[:,None] + time_increments

movavgd_etho_sees_home_face_when_home_trill = np.mean(etho_sees_home_face_ag[trills_home_start_1m].reshape(-1, trills_etho_start_1m.shape[1]//5, w),axis=2)
movavgd_home_sees_etho_face_when_home_trill = np.mean(home_sees_etho_face_ag[trills_home_start_1m].reshape(-1, trills_etho_start_1m.shape[1]//5, w),axis=2)
movavgd_etho_sees_home_face_when_etho_trill = np.mean(etho_sees_home_face_ag[trills_etho_start_1m].reshape(-1, trills_etho_start_1m.shape[1]//5, w),axis=2)
movavgd_home_sees_etho_face_when_etho_trill = np.mean(home_sees_etho_face_ag[trills_etho_start_1m].reshape(-1, trills_etho_start_1m.shape[1]//5, w),axis=2)
print(movavgd_etho_sees_home_face_when_home_trill.shape)
prob_etho_sees_home_face_when_home_trill = np.mean(movavgd_etho_sees_home_face_when_home_trill, axis=0)
prob_home_sees_etho_face_when_home_trill = np.mean(movavgd_home_sees_etho_face_when_home_trill, axis=0)
prob_etho_sees_home_face_when_etho_trill = np.mean(movavgd_etho_sees_home_face_when_etho_trill, axis=0)
prob_home_sees_etho_face_when_etho_trill = np.mean(movavgd_home_sees_etho_face_when_etho_trill, axis=0)
ci_prob_etho_sees_home_face_when_home_trill = stats.sem(movavgd_etho_sees_home_face_when_home_trill, axis=0, ddof=1) * 1.96
ci_prob_home_sees_etho_face_when_home_trill = stats.sem(movavgd_home_sees_etho_face_when_home_trill, axis=0, ddof=1) * 1.96
ci_prob_etho_sees_home_face_when_etho_trill = stats.sem(movavgd_etho_sees_home_face_when_etho_trill, axis=0, ddof=1) * 1.96
ci_prob_home_sees_etho_face_when_etho_trill = stats.sem(movavgd_home_sees_etho_face_when_etho_trill, axis=0, ddof=1) * 1.96


plt.plot(time_increments[1300:1700:w],prob_etho_sees_home_face_when_home_trill[1300//w:1700//w])
plt.plot(time_increments[1300:1700:w],prob_etho_sees_home_face_when_etho_trill[1300//w:1700//w])
plt.fill_between(time_increments[1300:1700:w],prob_etho_sees_home_face_when_home_trill[1300//w:1700//w] - ci_prob_etho_sees_home_face_when_home_trill[1300//w:1700//w], prob_etho_sees_home_face_when_home_trill[1300//w:1700//w] + ci_prob_etho_sees_home_face_when_home_trill[1300//w:1700//w], alpha = 0.5)
plt.fill_between(time_increments[1300:1700:w],prob_etho_sees_home_face_when_etho_trill[1300//w:1700//w] - ci_prob_etho_sees_home_face_when_etho_trill[1300//w:1700//w], prob_etho_sees_home_face_when_etho_trill[1300//w:1700//w] + ci_prob_etho_sees_home_face_when_etho_trill[1300//w:1700//w], alpha = 0.5)
plt.show()

plt.plot(time_increments[1300:1700:w],prob_home_sees_etho_face_when_home_trill[1300//w:1700//w])
plt.plot(time_increments[1300:1700:w],prob_home_sees_etho_face_when_etho_trill[1300//w:1700//w])
plt.fill_between(time_increments[1300:1700:w],prob_home_sees_etho_face_when_home_trill[1300//w:1700//w] - ci_prob_home_sees_etho_face_when_home_trill[1300//w:1700//w], prob_home_sees_etho_face_when_home_trill[1300//w:1700//w] + ci_prob_home_sees_etho_face_when_home_trill[1300//w:1700//w], alpha = 0.5)
plt.fill_between(time_increments[1300:1700:w],prob_home_sees_etho_face_when_etho_trill[1300//w:1700//w] - ci_prob_home_sees_etho_face_when_etho_trill[1300//w:1700//w], prob_home_sees_etho_face_when_etho_trill[1300//w:1700//w] + ci_prob_home_sees_etho_face_when_etho_trill[1300//w:1700//w], alpha = 0.5)
plt.show()


In [ ]:
print(np.max(distances_head[trills_home_start_10s]))
print(np.sum(distances_head > 2500) )

In [ ]:
def cones_intersect_vectorized(apexes1, axes1, heights1, angles1,
                              apexes2, axes2, heights2, angles2,
                              num_t=3, num_phi=4, eps=1e-10):
    """
    Vectorized intersection check for N cone pairs.
    Inputs: (N,3) for apexes/axes, (N,) for heights/angles.
    Returns: (N,) boolean array.
    """
    N = apexes1.shape[0]

    # Quick rejection: distance between apexes > sum of heights
    d_norm = np.linalg.norm(apexes2 - apexes1, axis=1)
    valid_dist = d_norm <= (heights1 + heights2)

    # Quick check: apexes inside other cone
    in1 = points_in_cone(apexes2, apexes1, axes1, heights1, angles1, eps)
    in2 = points_in_cone(apexes1, apexes2, axes2, heights2, angles2, eps)
    valid_apex = in1 | in2

    # Quick valid pairs (no need for surface sampling)
    quick_valid = valid_dist & valid_apex

    # For remaining pairs, generate surface sample points on cone1
    # Get perpendicular basis vectors for each cone1
    use_x = np.abs(axes1[:, 0]) < 0.5
    initial = np.zeros((N, 3))
    initial[use_x] = [1, 0, 0]
    initial[~use_x] = [0, 1, 0]

    a1 = initial - np.einsum('ij,ij->i', initial, axes1)[:, None] * axes1
    a1 = a1 / np.linalg.norm(a1, axis=1, keepdims=True)
    b1 = np.cross(axes1, a1)

    tan1 = np.tan(angles1)

    # Generate sample points on all cone1 surfaces
    t_vals = np.linspace(0, 1, num_t)  # (num_t,)
    phi_vals = np.linspace(0, 2*np.pi, num_phi, endpoint=False)  # (num_phi,)

    # Expand for broadcasting: (N, 1, 1, 3) and (1, num_t, 1, 1) etc.
    apex1_exp = apexes1[:, None, None, :]
    axes1_exp = axes1[:, None, None, :]
    a1_exp = a1[:, None, None, :]
    b1_exp = b1[:, None, None, :]
    h1_exp = heights1 if isinstance(heights1, float) or isinstance(heights1, int) else heights1[:, None, None, None]
    tan1_exp = tan1 if isinstance(tan1, float) else tan1[:, None, None, None]

    t_exp = t_vals[None, :, None, None]
    cos_phi = np.cos(phi_vals)[None, None, :, None]
    sin_phi = np.sin(phi_vals)[None, None, :, None]

    # P = apex + t*h*v + t*h*tanθ*(cosφ*a + sinφ*b)
    points = (apex1_exp +
              t_exp * h1_exp * axes1_exp +
              t_exp * h1_exp * tan1_exp * (cos_phi * a1_exp + sin_phi * b1_exp))

    # Reshape to (N * num_t * num_phi, 3)
    points = points.reshape(-1, 3)

    # Repeat cone2 parameters for each sample point
    apex2_rep = np.repeat(apexes2, num_t * num_phi, axis=0)
    axes1_rep = np.repeat(axes2, num_t * num_phi, axis=0)
    h2_rep = heights2 if isinstance(heights2, float) or isinstance(heights2, int) else np.repeat(heights2, num_t * num_phi)
    a2_rep = angles2 if isinstance(angles2, float) else np.repeat(angles2, num_t * num_phi)

    # Check if any sample point is in its corresponding cone2
    in_cone2 = points_in_cone(points, apex2_rep, axes1_rep, h2_rep, a2_rep, eps)
    any_in_cone2 = np.any(in_cone2.reshape(N, -1), axis=1)

    return quick_valid | any_in_cone2







test = cones_intersect_vectorized(apex_home, apex_etho, 3000, 5 * np.pi/180, axis_home, axis_etho, 3000, 5 * np.pi/180, int(3000/10), 32)









In [ ]:
xs = np.arange(-1500,1501,100)
ys = np.arange(-500,1501,100)
zs = np.arange(-500,1501,100)
x_allpoints, y_allpoints, z_allpoints = np.meshgrid(xs,ys,zs)
allpoints = np.concatenate((x_allpoints.flatten()[:,None,None], y_allpoints.flatten()[:,None,None], z_allpoints.flatten()[:,None,None]), axis = -1)
print(allpoints.shape)
home_sees_allpoints = points_in_cone(allpoints, apex_etho, axis_etho, 5 * np.pi/180, 3000)
etho_sees_allpoints = points_in_cone(allpoints, apex_home, axis_home, 5 * np.pi/180, 3000)

In [ ]:
print(np.sum(home_sees_etho, axis = 1)/home_sees_etho.shape[1])
print(np.sum(etho_sees_home, axis = 1)/etho_sees_home.shape[1])


In [ ]:

import os
import cv2
import matplotlib.pyplot as plt
import numpy as np
input_dir = f'/scratch/VideoTracking/Videos/260513/Etho/Output/videos_labeled_2d'
vidcaps = [cv2.VideoCapture(os.path.join(input_dir,f'output{i}.mp4')) for i in range(1,5)]
d = np.array([2337,2337,2335,2337])
d -= np.max(d)
f = 16825
for i, vidcap in enumerate(vidcaps):
    vidcap.set(cv2.CAP_PROP_POS_FRAMES, f + d[i])
    success, frame = vidcap.read()
    if success:
        plt.imshow(frame[:,:,::-1])
        plt.axis('off')
        plt.tight_layout()
        plt.subplots_adjust(left=0, right=1, top=1, bottom=0) 
        plt.savefig(f'output_etho{i}.png', dpi=300, bbox_inches='tight', pad_inches=0)
        plt.cla()
        plt.close()

input_dir = f'/scratch/VideoTracking/Videos/260513/Home/Output/videos_labeled_2d'
vidcaps = [cv2.VideoCapture(os.path.join(input_dir,f'output{i}.mp4')) for i in range(1,7)]
d = np.array([2365,2365,2364,2364,2364,2365])
d -= np.max(d)
f += 28
for i, vidcap in enumerate(vidcaps):
    vidcap.set(cv2.CAP_PROP_POS_FRAMES, f + d[i])
    success, frame = vidcap.read()
    if success:
        plt.imshow(frame[:,:,::-1])
        plt.axis('off')
        plt.tight_layout()
        plt.subplots_adjust(left=0, right=1, top=1, bottom=0) 
        plt.savefig(f'output_home{i}.png', dpi=300, bbox_inches='tight', pad_inches=0)
        plt.cla()
        plt.close()

vidcap = cv2.VideoCapture(f'/scratch/VideoTracking/Videos/260513/Output_combined/optimized_combined.mp4')
f -= 2365
vidcap.set(cv2.CAP_PROP_POS_FRAMES, f)
success, frame = vidcap.read()
if success:
    plt.imshow(frame[:,:,::-1])
    plt.axis('off')
    plt.tight_layout()
    plt.subplots_adjust(left=0, right=1, top=1, bottom=0) 
    plt.savefig(f'output_combined.png', dpi=300, bbox_inches='tight', pad_inches=0)
    plt.cla()
    plt.close()


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
max_dist_etho = np.linalg.norm(etho_dimensions)
max_dist_home = np.linalg.norm(home_dimensions)
fs = 26
lfs = 22
tfs = 22
lgfs = 22
lw = 3
color_etho = '#964ea3'
color_home = '#ff7f00'
color_phee = '#e41a1c'
color_twitter = '#a65628'
color_trill = '#377eb8'
def simpleaxis(ax):
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.get_xaxis().tick_bottom()
    ax.get_yaxis().tick_left()
    for spine in ax.spines.values():
        spine.set_linewidth(2)        # thicker axis lines

    # Lengthen and thicken ticks
    ax.tick_params(axis='both', which='major',
                length=10,            # tick length in points
                width=2,              # tick width in points
                direction='out')      # 'in', 'out', or 'inout'

    # If you want longer minor ticks too:
    ax.tick_params(axis='both', which='minor',
                length=6, width=1)


fig = plt.figure(figsize=(20,15),constrained_layout=True,)
fig.set_constrained_layout_pads(hspace=0.1, wspace=0.05)
gs = gridspec.GridSpec(nrows=4, ncols=6, figure=fig)

bin_step = 30
n_ticks = 4
indices = (idx_head, idx_leftear, idx_rightear)
points = all_points_3d - offset
dimensions = etho_dimensions
positions = np.mean(points[:,np.r_[indices]], axis = 1)

ax1 = fig.add_subplot(gs[0, 0:2])
ax2 = fig.add_subplot(gs[0, 2:4])
ax3 = fig.add_subplot(gs[0, 4:6])

bins_x = np.arange(0, dimensions[0] + bin_step, bin_step)
bins_y = np.arange(0, dimensions[1] + bin_step, bin_step)
bins_z = np.arange(0, dimensions[2] + bin_step, bin_step)

heatmap, _, _ = np.histogram2d(positions[:,1],positions[:,0],bins = (bins_y, bins_x))
print(heatmap.shape)
ax1.imshow(heatmap[::-1,:], cmap='viridis', interpolation='nearest', aspect='equal')
xticks = np.arange(0, heatmap.shape[1] + 1, heatmap.shape[1]/n_ticks)
yticks = np.arange(0, heatmap.shape[0] + 1, heatmap.shape[0]/n_ticks)
ax1.set_xticks(xticks - 0.5, labels = (xticks*bin_step/10).astype(int), fontsize = tfs)
ax1.set_yticks(heatmap.shape[0] - yticks - 0.5, labels = (yticks*bin_step/10).astype(int), fontsize = tfs)
ax1.set_xlabel('x position (cm)', fontsize = lfs)
ax1.set_ylabel('y position (cm)', fontsize = lfs)

heatmap, _, _ = np.histogram2d(positions[:,2],positions[:,0],bins = (bins_z, bins_x))
ax2.imshow(heatmap[::-1,:], cmap='viridis', interpolation='nearest', aspect='equal')
yticks = np.arange(0, heatmap.shape[0] + 1, heatmap.shape[0]/n_ticks)
ax2.set_xticks(xticks - 0.5, labels = (xticks*bin_step/10).astype(int), fontsize = tfs)
ax2.set_yticks(heatmap.shape[0] - yticks - 0.5, labels = (yticks*bin_step/10).astype(int), fontsize = tfs)
ax2.set_xlabel('x position (cm)', fontsize = lfs)
ax2.set_ylabel('z position (cm)', fontsize = lfs)

heatmap, _, _ = np.histogram2d(positions[:,2],positions[:,1],bins = (bins_z, bins_y))
ax3.imshow(heatmap[::-1,:], cmap='viridis', interpolation='nearest', aspect='equal')
xticks = np.arange(0, heatmap.shape[1] + 1, heatmap.shape[1]/n_ticks)
ax3.set_xticks(xticks - 0.5, labels = (xticks*bin_step/10).astype(int), fontsize = tfs)
ax3.set_yticks(heatmap.shape[0] - yticks - 0.5, labels = (yticks*bin_step/10).astype(int), fontsize = tfs)
ax3.set_xlabel('y position (cm)', fontsize = lfs)
ax3.set_ylabel('z position (cm)', fontsize = lfs)

points = all_points_3d_home - offset_home
dimensions = home_dimensions
positions = np.mean(points[:,np.r_[indices]], axis = 1)

ax4 = fig.add_subplot(gs[1, 0:2])
ax5 = fig.add_subplot(gs[1, 2:4])
ax6 = fig.add_subplot(gs[1, 4:6])

bins_x = np.arange(0, dimensions[0] + bin_step, bin_step)
bins_y = np.arange(0, dimensions[1] + bin_step, bin_step)
bins_z = np.arange(0, dimensions[2] + bin_step, bin_step)

heatmap, _, _ = np.histogram2d(positions[:,1],positions[:,0],bins = (bins_y, bins_x))
print(heatmap.shape)
ax4.imshow(heatmap[::-1,:], cmap='viridis', interpolation='nearest', aspect='equal')
xticks = np.arange(0, heatmap.shape[1] + 1, heatmap.shape[1]/n_ticks)
yticks = np.arange(0, heatmap.shape[0] + 1, heatmap.shape[0]/n_ticks)
ax4.set_xticks(xticks - 0.5, labels = (xticks*bin_step/10).astype(int), fontsize = tfs)
ax4.set_yticks(heatmap.shape[0] - yticks - 0.5, labels = (yticks*bin_step/10).astype(int), fontsize = tfs)
ax4.set_xlabel('x position (cm)', fontsize = lfs)
ax4.set_ylabel('y position (cm)', fontsize = lfs)

heatmap, _, _ = np.histogram2d(positions[:,2],positions[:,0],bins = (bins_z, bins_x))
ax5.imshow(heatmap[::-1,:], cmap='viridis', interpolation='nearest', aspect='equal')
yticks = np.arange(0, heatmap.shape[0] + 1, heatmap.shape[0]/n_ticks)
ax5.set_xticks(xticks - 0.5, labels = (xticks*bin_step/10).astype(int), fontsize = tfs)
ax5.set_yticks(heatmap.shape[0] - yticks - 0.5, labels = (yticks*bin_step/10).astype(int), fontsize = tfs)
ax5.set_xlabel('x position (cm)', fontsize = lfs)
ax5.set_ylabel('z position (cm)', fontsize = lfs)

heatmap, _, _ = np.histogram2d(positions[:,2],positions[:,1],bins = (bins_z, bins_y))
ax6.imshow(heatmap[::-1,:], cmap='viridis', interpolation='nearest', aspect='equal')
xticks = np.arange(0, heatmap.shape[1] + 1, heatmap.shape[1]/n_ticks)
ax6.set_xticks(xticks - 0.5, labels = (xticks*bin_step/10).astype(int), fontsize = tfs)
ax6.set_yticks(heatmap.shape[0] - yticks - 0.5, labels = (yticks*bin_step/10).astype(int), fontsize = tfs)
ax6.set_xlabel('y position (cm)', fontsize = lfs)
ax6.set_ylabel('z position (cm)', fontsize = lfs)

ax7 = fig.add_subplot(gs[2, 0:2])
ax8 = fig.add_subplot(gs[2, 2:4])
ax9 = fig.add_subplot(gs[2, 4:6])

positions1 = np.mean(all_points_3d[:,np.r_[indices]], axis = 1)
positions2 = np.mean(all_points_3d_home[:,np.r_[indices]], axis = 1)

distances = np.sqrt(np.einsum('ij,ij->i',positions1 - positions2, positions1 - positions2))
counts, bins = np.histogram(distances,bins=np.arange(0,2501,100))
ax7.plot((bins[:-1] + bins[1:])/2, counts/np.sum(counts), c = '#4daf4a', lw = lw)
ax7.set_ylabel('Probability density', fontsize = lfs)
ax7.set_xlabel('Distance (cm)', fontsize = lfs)
xticks = np.arange(0,2401,800)
ax7.set_xticks(xticks,labels=(xticks/10).astype(int),fontsize=tfs)
yticks = np.arange(0,0.16,0.05)
ax7.set_yticks(yticks,labels=np.round(yticks,2),fontsize=tfs)
ax7.set_ylim((0,0.15))
ax7.set_title('B \u2194 K', size=fs)
ax7 = simpleaxis(ax7)

positions1 = np.mean(all_points_3d[:,np.r_[indices]], axis = 1)
positions2 = np.mean(closest_to_etho[:,np.r_[indices]], axis = 1)

distances = np.sqrt(np.einsum('ij,ij->i',positions1 - positions2, positions1 - positions2))
counts, bins = np.histogram(distances,bins=np.linspace(0,max_dist_etho, 20))
ax8.plot((bins[:-1] + bins[1:])/(2*max_dist_etho), counts/np.sum(counts), c = '#4daf4a', lw = lw)
ax8.set_xlabel('Normalized distance', fontsize = lfs)
xticks = np.arange(0,1.1,0.25)
ax8.set_xticks(xticks,labels=xticks,fontsize=tfs)
ax8.set_xlim((0,1))
yticks = np.arange(0,0.31,0.1)
ax8.set_yticks(yticks,labels=np.round(yticks,2),fontsize=tfs)
ax8.set_ylim((0,0.3))
ax8.set_title('B \u2194 Closest to K', size=fs)
ax8 = simpleaxis(ax8)

positions1 = np.mean(all_points_3d_home[:,np.r_[indices]], axis = 1)
positions2 = np.mean(closest_to_home[:,np.r_[indices]], axis = 1)

distances = np.sqrt(np.einsum('ij,ij->i',positions1 - positions2, positions1 - positions2))
counts, bins = np.histogram(distances,bins=np.linspace(0,max_dist_home, 20))
ax9.plot((bins[:-1] + bins[1:])/(2*max_dist_home), counts/np.sum(counts), c = '#4daf4a', lw = lw)
ax9.set_xlabel('Normalized distance', fontsize = lfs)
xticks = np.arange(0,1.1,0.25)
ax9.set_xticks(xticks,labels=xticks,fontsize=tfs)
ax9.set_xlim((0,1))
yticks = np.arange(0,0.19,0.06)
ax9.set_yticks(yticks,labels=np.round(yticks,2),fontsize=tfs)
ax9.set_ylim((0,0.18))
ax9.set_title('K \u2194 Closest to B', size=fs)
ax9 = simpleaxis(ax9)

distance_bins = np.arange(0,2501,125)
indices = (idx_head, idx_rightear, idx_leftear)
positions1 = np.mean(all_points_3d[:,np.r_[indices]], axis = 1)
positions2 = np.mean(all_points_3d_home[:,np.r_[indices]], axis = 1)
positions_closest_to_home = np.mean(closest_to_home[:,np.r_[indices]], axis = 1)
positions_closest_to_etho = np.mean(closest_to_etho[:,np.r_[indices]], axis = 1)
distances_head = np.linalg.norm(positions1-positions2, axis = 1)
indices_bins = [np.where(np.logical_and(distances_head >= bin1,distances_head < bin2))[0] for bin1,bin2 in zip(distance_bins[:-1], distance_bins[1:])]
distances_head_closest_to_home = np.linalg.norm(positions2-positions_closest_to_home, axis = 1)
indices_bins_closest_to_home = [np.where(np.logical_and(distances_head_closest_to_home >= bin1,distances_head_closest_to_home < bin2))[0] for bin1,bin2 in zip(distance_bins[:-1], distance_bins[1:])]
distances_head_closest_to_etho = np.linalg.norm(positions1-positions_closest_to_etho, axis = 1)
indices_bins_to_etho = [np.where(np.logical_and(distances_head_closest_to_etho >= bin1,distances_head_closest_to_etho < bin2))[0] for bin1,bin2 in zip(distance_bins[:-1], distance_bins[1:])]


ax10 = fig.add_subplot(gs[3, 0:2])
ax11 = fig.add_subplot(gs[3, 2:4])
ax12 = fig.add_subplot(gs[3, 4:6])
etho_sees_home_ag_per_distance_bin = [np.sum(etho_sees_home_ag[indices])/indices.size for indices in indices_bins]
home_sees_etho_ag_per_distance_bin = [np.sum(home_sees_etho_ag[indices])/indices.size for indices in indices_bins]
etho_sees_home_face_ag_per_distance_bin = [np.sum(etho_sees_home_face_ag[indices])/indices.size for indices in indices_bins]
home_sees_etho_face_ag_per_distance_bin = [np.sum(home_sees_etho_face_ag[indices])/indices.size for indices in indices_bins]
etho_sees_home_body_ag_distance_bin = [np.sum(etho_sees_home_body_ag[indices])/indices.size for indices in indices_bins]
home_sees_etho_body_ag_per_distance_bin = [np.sum(home_sees_etho_body_ag[indices])/indices.size for indices in indices_bins]
ax10.plot((distance_bins[1:] + distance_bins[:-1])/2,etho_sees_home_ag_per_distance_bin, label='B', c = color_etho, lw = lw)
ax10.plot((distance_bins[1:] + distance_bins[:-1])/2,home_sees_etho_ag_per_distance_bin, label='K', c = color_home, lw = lw)
ax10.set_ylabel('Probability', fontsize = lfs)
ax10.set_xlabel('Distance (cm)', fontsize = lfs)
xticks = np.arange(0,2501,500)
ax10.set_xticks(xticks,labels=(xticks/10).astype(int),fontsize=tfs)
yticks = np.arange(0,0.19,0.06)
ax10.set_yticks(yticks,labels=np.round(yticks,2),fontsize=tfs)
ax10.set_ylim((0,0.20))
ax10.set_title('Gaze to other', size=fs)
ax10.legend(prop={'size': lgfs}, loc = 'upper right', frameon=False)
ax10 = simpleaxis(ax10)

ax11.plot((distance_bins[1:] + distance_bins[:-1])/2,etho_sees_home_face_ag_per_distance_bin, c = color_etho, lw = lw)
ax11.plot((distance_bins[1:] + distance_bins[:-1])/2,home_sees_etho_face_ag_per_distance_bin, c = color_home, lw = lw)
ax11.set_xlabel('Distance (cm)', fontsize = lfs)
ax11.set_xticks(xticks,labels=(xticks/10).astype(int),fontsize=tfs)
yticks = np.arange(0,0.07,0.02)
ax11.set_yticks(yticks,labels=np.round(yticks,2),fontsize=tfs)
ax11.set_ylim((0,0.06))
ax11.set_title('Gaze to other\'s head', size=fs)
ax11 = simpleaxis(ax11)

ax12.plot((distance_bins[1:] + distance_bins[:-1])/2,etho_sees_home_body_ag_distance_bin, c = color_etho, lw = lw)
ax12.plot((distance_bins[1:] + distance_bins[:-1])/2,home_sees_etho_body_ag_per_distance_bin, c = color_home, lw = lw)
ax12.set_xlabel('Distance (cm)', fontsize = lfs)
ax12.set_xticks(xticks,labels=(xticks/10).astype(int),fontsize=tfs)
yticks = np.arange(0,0.1,0.03)
ax12.set_yticks(yticks,labels=np.round(yticks,2),fontsize=tfs)
ax12.set_ylim((0,0.1))
ax12.set_title('Gaze to other\'s trunk', size=fs)
ax12 = simpleaxis(ax12)
fig.savefig('/scratch/VideoTracking/MarmoPose/analysis/Position/Fig3.png',dpi=300)



In [ ]:
print(distance_bins.shape)
print(distance_bins)
print(len(twitter_etho_start_per_distance_bin))

In [ ]:
fig = plt.figure(figsize=(20,15),constrained_layout=True,)
fig.set_constrained_layout_pads(hspace=0.05, wspace=0.05)
gs = gridspec.GridSpec(nrows=3, ncols=6, figure=fig)
distance_bins = np.arange(0,2501,250)

ax1 = fig.add_subplot(gs[0, 0:3])
ax2 = fig.add_subplot(gs[0, 3:6])
ax1.plot((distance_bins[1:] + distance_bins[:-1])/2,trill_etho_start_per_distance_bin,label = 'Trills', c = color_trill, lw = lw)
ax1.plot((distance_bins[1:] + distance_bins[:-1])/2,twitter_etho_start_per_distance_bin,label = 'Twitters', c = color_twitter, lw = lw)
ax1.plot((distance_bins[1:] + distance_bins[:-1])/2,phee_etho_start_per_distance_bin,label = 'Phees', c = color_phee, lw = lw)
xticks = np.arange(0,2501,500)
ax1.set_xlabel('Distance (cm)', fontsize = lfs)
ax1.set_xticks(xticks,labels=(xticks/10).astype(int),fontsize=tfs)
ax1.set_ylabel('Rate', fontsize = lfs)
yticks = np.arange(0,0.009,0.002)
ax1.set_yticks(yticks,labels=np.round(yticks,3),fontsize=tfs)
ax1.set_ylim((0,0.008))
ax1.set_title('B vocalization rate', size=fs)
ax1.legend(prop={'size': lgfs}, frameon=False)
simpleaxis(ax1)

ax2.plot(distance_bins[1:],trill_home_start_per_distance_bin, c = color_trill, lw = lw)
ax2.plot(distance_bins[1:],twitter_home_start_per_distance_bin, c = color_twitter, lw = lw)
ax2.plot(distance_bins[1:],phee_home_start_per_distance_bin, c = color_phee, lw = lw)
ax2.set_xlabel('Distance (cm)', fontsize = lfs)
ax2.set_xticks(xticks,labels=(xticks/10).astype(int),fontsize=tfs)
yticks = np.arange(0,0.005,0.001)
ax2.set_yticks(yticks,labels=np.round(yticks,3),fontsize=tfs)
ax2.set_ylim((0,0.004))
ax2.set_title('K vocalization rate', size=fs)
ax2.legend(prop={'size': lgfs}, frameon=False)
simpleaxis(ax2)

ax3 = fig.add_subplot(gs[1, 0:2])
ax4 = fig.add_subplot(gs[1, 2:4])
ax5 = fig.add_subplot(gs[1, 4:6])

ax3.plot(time_increments,mean_distance_post_trill_etho, label= 'B Trills', c = color_etho, lw = lw)
ax3.fill_between(time_increments, mean_distance_post_trill_etho - ci_distance_post_trill_etho, mean_distance_post_trill_etho + ci_distance_post_trill_etho, alpha=0.5, color = color_etho, lw = lw)
ax3.plot(time_increments,mean_distance_post_trill_home, label= 'K Trills', c = color_home, lw = lw)
ax3.fill_between(time_increments, mean_distance_post_trill_home - ci_distance_post_trill_home, mean_distance_post_trill_home + ci_distance_post_trill_home, alpha=0.5, color = color_home, lw = lw)
ax3.set_xlim((-1500,1500))
ax3.set_xlabel('Time to trill onset (s)', fontsize = lfs)
xticks = np.arange(-1500,1501,750)
ax3.set_xticks(xticks,labels=(xticks/25).astype(int),fontsize=tfs)
ax3.set_ylabel('Distance (cm)', fontsize = lfs)
yticks = np.arange(700,1501,400)
ax3.set_yticks(yticks,labels=np.round(yticks,3),fontsize=tfs)
ax3.set_ylim((700,1500))
ax3.set_title('Distance B \u2194 K\nAt trill onset', size=fs)
ax3.legend(loc='upper right', bbox_to_anchor=(1.05, 1.05), prop={'size': lgfs}, frameon=False)
simpleaxis(ax3)

ax4.plot(time_increments,mean_distance_closest_home_post_trill_etho, c = color_etho, lw = lw)
ax4.fill_between(time_increments, mean_distance_closest_home_post_trill_etho - ci_distance_closest_home_post_trill_etho, mean_distance_closest_home_post_trill_etho + ci_distance_closest_home_post_trill_etho, alpha=0.5, color = color_etho, lw = lw)
ax4.plot(time_increments,mean_distance_closest_home_post_trill_home, c = color_home, lw = lw)
ax4.fill_between(time_increments, mean_distance_closest_home_post_trill_home - ci_distance_closest_home_post_trill_home, mean_distance_closest_home_post_trill_home + ci_distance_closest_home_post_trill_home, alpha=0.5, color = color_home, lw = lw)
ax4.set_xlim((-1500,1500))
ax4.set_xlabel('Time to trill onset (s)', fontsize = lfs)
xticks = np.arange(-1500,1501,750)
ax4.set_xticks(xticks,labels=(xticks/25).astype(int),fontsize=tfs)
yticks = np.arange(500,1001,250)
ax4.set_yticks(yticks,labels=np.round(yticks,3),fontsize=tfs)
ax4.set_ylim((500,1000))
ax4.set_title('Distance B \u2194 Closest to K\nAt trill onset', size=fs)
simpleaxis(ax4)


ax5.plot(time_increments,mean_distance_closest_etho_post_trill_etho, c = color_etho, lw = lw)
ax5.fill_between(time_increments, mean_distance_closest_etho_post_trill_etho - ci_distance_closest_etho_post_trill_etho, mean_distance_closest_etho_post_trill_etho + ci_distance_closest_etho_post_trill_etho, alpha=0.5, color = color_etho, lw = lw)
ax5.plot(time_increments,mean_distance_closest_etho_post_trill_home, c = color_home, lw = lw)
ax5.fill_between(time_increments, mean_distance_closest_etho_post_trill_home - ci_distance_closest_etho_post_trill_home, mean_distance_closest_etho_post_trill_home + ci_distance_closest_etho_post_trill_home, alpha=0.5, color = color_home, lw = lw)
ax5.set_xlim((-1500,1500))
ax5.set_xlabel('Time to trill onset (s)', fontsize = lfs)
xticks = np.arange(-1500,1501,750)
ax5.set_xticks(xticks,labels=(xticks/25).astype(int),fontsize=tfs)
yticks = np.arange(500,801,150)
ax5.set_yticks(yticks,labels=np.round(yticks,3),fontsize=tfs)
ax5.set_ylim((500,800))
ax5.set_title('Distance K \u2194 Closest to B\nAt trill onset', size=fs)
simpleaxis(ax5)

ax6 = fig.add_subplot(gs[2, 0:3])
ax7 = fig.add_subplot(gs[2, 3:6])

ax6.plot(time_increments[1300:1700:w],prob_etho_sees_home_face_when_etho_trill[1300//w:1700//w], label= 'B Trills', c = color_etho, lw = lw)
ax6.fill_between(time_increments[1300:1700:w],prob_etho_sees_home_face_when_etho_trill[1300//w:1700//w] - ci_prob_etho_sees_home_face_when_etho_trill[1300//w:1700//w], prob_etho_sees_home_face_when_etho_trill[1300//w:1700//w] + ci_prob_etho_sees_home_face_when_etho_trill[1300//w:1700//w], alpha = 0.5, color = color_etho, lw = lw)
ax6.plot(time_increments[1300:1700:w],prob_etho_sees_home_face_when_home_trill[1300//w:1700//w], label= 'K Trills', c = color_home, lw = lw)
ax6.fill_between(time_increments[1300:1700:w],prob_etho_sees_home_face_when_home_trill[1300//w:1700//w] - ci_prob_etho_sees_home_face_when_home_trill[1300//w:1700//w], prob_etho_sees_home_face_when_home_trill[1300//w:1700//w] + ci_prob_etho_sees_home_face_when_home_trill[1300//w:1700//w], alpha = 0.5, color = color_home, lw = lw)
ax6.set_xlabel('Time to trill onset (s)', fontsize = lfs)
ax6.set_xlim((-200,200))
xticks = np.arange(-200,201,100)
ax6.set_xticks(xticks,labels=(xticks/25).astype(int),fontsize=tfs)
ax6.set_ylabel('Probability', fontsize = lfs)
yticks = np.arange(0,0.09,0.04)
ax6.set_yticks(yticks,labels=np.round(yticks,3),fontsize=tfs)
ax6.set_title('Probability that B sees K\nAt trill onset', size=fs)
ax6.legend(loc='upper left', bbox_to_anchor=(-0.02, 1.05), prop={'size': lgfs}, frameon=False)
simpleaxis(ax6)

ax7.plot(time_increments[1300:1700:w],prob_home_sees_etho_face_when_etho_trill[1300//w:1700//w], c = color_etho, lw = lw)
ax7.fill_between(time_increments[1300:1700:w],prob_home_sees_etho_face_when_etho_trill[1300//w:1700//w] - ci_prob_home_sees_etho_face_when_etho_trill[1300//w:1700//w], prob_home_sees_etho_face_when_etho_trill[1300//w:1700//w] + ci_prob_home_sees_etho_face_when_etho_trill[1300//w:1700//w], alpha = 0.5, color = color_etho, lw = lw)
ax7.plot(time_increments[1300:1700:w],prob_home_sees_etho_face_when_home_trill[1300//w:1700//w], c = color_home, lw = lw)
ax7.fill_between(time_increments[1300:1700:w],prob_home_sees_etho_face_when_home_trill[1300//w:1700//w] - ci_prob_home_sees_etho_face_when_home_trill[1300//w:1700//w], prob_home_sees_etho_face_when_home_trill[1300//w:1700//w] + ci_prob_home_sees_etho_face_when_home_trill[1300//w:1700//w], alpha = 0.5, color = color_home, lw = lw)
ax7.set_xlabel('Time to trill onset (s)', fontsize = lfs)
ax7.set_xlim((-200,200))
xticks = np.arange(-200,201,100)
ax7.set_xticks(xticks,labels=(xticks/25).astype(int),fontsize=tfs)
ax7.set_ylabel('Probability', fontsize = lfs)
yticks = np.arange(0,0.07,0.03)
ax7.set_yticks(yticks,labels=np.round(yticks,3),fontsize=tfs)
ax7.set_title('Probability that K sees B\nAt trill onset', size=fs)
ax7.legend(prop={'size': lgfs}, frameon=False)
simpleaxis(ax7)

fig.savefig('/scratch/VideoTracking/MarmoPose/analysis/Position/Fig4.png',dpi=300)
fig.show()


In [ ]:
print(time_increments[1300:1701:w])